#### Decodificando la Ley: Clasificación Inteligente y Búsqueda Semántica de Jurisprudencia Argentina

##### Descarga del dataset completo

Instalamos primero las librerías que necesitaremos y luego las importamos

In [ ]:
# %pip install huggingface_hub datasets pandas

In [ ]:
# Importamos librerías
from huggingface_hub import snapshot_download
import pandas as pd
import os

# # Variables de configuración
# DATASET_PATH = "datos/full-dataset"
# RANDOM_SEED = 42

DATASET_PATH = "gitrepo\\datos\\dataset_sample.jsonl.gz"


Descargamos el dataset completo (~2.5GB)

In [ ]:
# # Importamos librerías
# from huggingface_hub import snapshot_download, login
# import pandas as pd
# import os

# # Autenticación
# HF_TOKEN = ""  # Reemplaza con tu token de Hugging Face
# login(token=HF_TOKEN)

# # Variables de configuración
# DATASET_PATH = "datos/full-dataset"
# RANDOM_SEED = 42

# # Si el dataset ya existe localmente, no lo descargamos de nuevo
# if os.path.isdir(DATASET_PATH):
#     print("El dataset ya ha sido descargado previamente.")
# else:
#     print("Descargando el dataset...")

#     snapshot_download(
#         repo_id="marianbasti/jurisprudencia-Argentina-SAIJ",
#         repo_type="dataset",
#         local_dir=DATASET_PATH,
#         token=HF_TOKEN
#     )

#     print("Descarga finalizada.")

##### Lectura de los datasets

Dataset completo

In [ ]:
# df = pd.read_json(
#     os.path.join(DATASET_PATH, "dataset.jsonl"),
#     lines=True
# )

Dataset de ejemplo (muestra del 1% del dataset completo)

In [ ]:
from huggingface_hub import snapshot_download
import pandas as pd
import os
from pathlib import Path

pd.set_option("display.max_rows", None)      # muestra todas las filas
pd.set_option("display.max_columns", None)   # opcional: todas las columnas
pd.set_option("display.max_colwidth", None)  #

DATASET_PATH = Path("datos") / "dataset_sample.jsonl.gz"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset en {DATASET_PATH.resolve()}. "
        "Asegurate de que el archivo exista en la carpeta `datos`."
    )

df_sample = pd.read_json(
    DATASET_PATH,
    lines=True,
    compression="gzip"
)

# df_sample = pd.read_json(
#     "dataset_sample.jsonl.gz",
#     lines=True,
#     compression="gzip"
# )

In [ ]:
df_sample.head()

In [ ]:


df_sample.head()

#analisis de valores nuilos por columna 
n_filas = len(df_sample)

df_nulos = pd.DataFrame({
    "columna": df_sample.columns,
    "nulos": df_sample.isnull().sum().values,
    "no_nulos": df_sample.notna().sum().values,
    "pct_nulos": df_sample.isnull().sum().values / n_filas * 100,
    "pct_completos": df_sample.notna().sum().values / n_filas * 100,
}).assign(
    tiene_nulos=lambda d: d["nulos"] > 0
).sort_values("pct_nulos", ascending=False).reset_index(drop=True)

# Solo columnas con al menos un valor nulo
df_nulos[df_nulos["tiene_nulos"]]

#dataset sin columnas vacias 
def es_vacio(serie: pd.Series) -> pd.Series:
    """Marca valores nulos (NaN/None) y strings vacíos como vacíos."""
    vacios = serie.isna()
    if serie.dtype == object:
        vacios = vacios | (serie == "")
    return vacios


UMBRAL_VACIOS = 99.97  # redondeado a 2 decimales (ej. 99.965706% -> 99.97%)

pct_vacios = df_sample.apply(lambda col: es_vacio(col).mean() * 100)
columnas_removidas = pct_vacios[pct_vacios.round(2) >= UMBRAL_VACIOS].index.tolist()

df_columnas_removidas = (
    pct_vacios[columnas_removidas]
    .rename("pct_vacios")
    .reset_index()
    .rename(columns={"index": "columna"})
    .sort_values("pct_vacios", ascending=False)
    .reset_index(drop=True)
)

df_sample_limpio = df_sample.drop(columns=columnas_removidas)

print(f"Umbral aplicado: >= {UMBRAL_VACIOS}% vacíos (redondeado)")
print(f"Columnas removidas ({len(columnas_removidas)}):")
print(columnas_removidas)
print(f"\nShape original: {df_sample.shape}")
print(f"Shape limpio:   {df_sample_limpio.shape}")

df_columnas_removidas


df_sample_limpio.columns.to_list()


n_filas = len(df_sample_limpio)

df_nulos = pd.DataFrame({
    "columna": df_sample_limpio.columns,
    "nulos": df_sample_limpio.isnull().sum().values,
    "no_nulos": df_sample_limpio.notna().sum().values,
    "pct_nulos": df_sample_limpio.isnull().sum().values / n_filas * 100,
    "pct_completos": df_sample_limpio.notna().sum().values / n_filas * 100,
}).assign(
    tiene_nulos=lambda d: d["nulos"] > 0
).sort_values("pct_nulos", ascending=False).reset_index(drop=True)

# Solo columnas con al menos un valor nulo
df_nulos[df_nulos["tiene_nulos"]]


total_filas = len(df_sample_limpio)
pais_con_valor = df_sample_limpio["pais"].notna().sum()
pais_nulos = df_sample_limpio["pais"].isna().sum()

print(f"Total filas:        {total_filas}")
print(f"Con país informado: {pais_con_valor} ({pais_con_valor / total_filas * 100:.2f}%)")
print(f"Nulos (sin país):   {pais_nulos} ({pais_nulos / total_filas * 100:.2f}%)")
print(f"\nVerificación: value_counts().sum() = {df_sample_limpio['pais'].value_counts().sum()}")

df_sample_limpio["pais"].value_counts(dropna=False)


total_filas = len(df_sample_limpio)
provincia_con_valor = df_sample_limpio["provincia"].notna().sum()
provincia_nulos = df_sample_limpio["provincia"].isna().sum()

print(f"Total filas:            {total_filas}")
print(f"Con provincia informada: {provincia_con_valor} ({provincia_con_valor / total_filas * 100:.2f}%)")
print(f"Nulos (sin provincia):   {provincia_nulos} ({provincia_nulos / total_filas * 100:.2f}%)")
print(f"\nVerificación: value_counts().sum() = {df_sample_limpio['provincia'].value_counts().sum()}")

df_sample_limpio["provincia"].value_counts(dropna=False)


df_sample_limpio.pais.value_counts()


def es_vacio(serie: pd.Series) -> pd.Series:
    vacios = serie.isna()
    if serie.dtype == object:
        vacios = vacios | (serie == "")
    return vacios

sin_provincia = df_sample_limpio[df_sample_limpio["provincia"].isna()]
n = len(sin_provincia)

columnas_con_datos = []
for col in sin_provincia.columns:
    no_vacios = (~es_vacio(sin_provincia[col])).sum()
    if no_vacios > 0:
        columnas_con_datos.append({
            "columna": col,
            "no_vacios": no_vacios,
            "pct_filas": no_vacios / n * 100,
        })

df_cols_sin_provincia = (
    pd.DataFrame(columnas_con_datos)
    .sort_values("no_vacios", ascending=False)
    .reset_index(drop=True)
)

df_cols_sin_provincia


sin_provincia.head()



sin_provincia = es_vacio(df_sample_limpio["provincia"])
sin_fecha = es_vacio(df_sample_limpio["fecha"])
sin_fecha_alta = es_vacio(df_sample_limpio["fecha-alta"])

dropeables = sin_provincia & sin_fecha_alta
resto = sin_provincia & ~sin_fecha_alta

print(f"Dropeables: {dropeables.sum()}")
print(f"Resto sin provincia: {resto.sum()}")
print(f"Quedarían en total: {len(df_sample_limpio) - dropeables.sum()}")

df_sample_limpio[dropeables].shape
df_sample_limpio[resto][["provincia", "fecha", "fecha-alta", "pais", "materia", "texto"]]


df_filtrado = df_sample_limpio[~dropeables]


df_filtrado.shape


total_filas = len(df_filtrado)
provincia_con_valor = df_filtrado["provincia"].notna().sum()
provincia_nulos = df_filtrado["provincia"].isna().sum()

print(f"Total filas:            {total_filas}")
print(f"Con provincia informada: {provincia_con_valor} ({provincia_con_valor / total_filas * 100:.2f}%)")
print(f"Nulos (sin provincia):   {provincia_nulos} ({provincia_nulos / total_filas * 100:.2f}%)")
print(f"\nVerificación: value_counts().sum() = {df_filtrado['provincia'].value_counts().sum()}")

df_filtrado["provincia"].value_counts(dropna=False)


provincias_extranjeras = [
    "San José de Costa Rica",
    "Ginebra",
    "Bajo Rin",
    "Madrid",
]

df_filtrado[df_filtrado["provincia"].isin(provincias_extranjeras)][
    ["provincia", "pais", "jurisdiccion", "materia", "caratula"]
]


provincias_extranjeras = [
    "San José de Costa Rica",
    "Ginebra",
    "Bajo Rin",
    "Madrid",
]

df_filtrado_arg = df_filtrado[~df_filtrado["provincia"].isin(provincias_extranjeras)]

total_filas = len(df_filtrado_arg)
provincia_con_valor = df_filtrado_arg["provincia"].notna().sum()
provincia_nulos = df_filtrado_arg["provincia"].isna().sum()

print(f"Filas removidas (provincias extranjeras): {len(df_filtrado) - total_filas}")
print(f"Total filas:            {total_filas}")
print(f"Con provincia informada: {provincia_con_valor} ({provincia_con_valor / total_filas * 100:.2f}%)")
print(f"Nulos (sin provincia):   {provincia_nulos} ({provincia_nulos / total_filas * 100:.2f}%)")
print(f"\nVerificación: value_counts().sum() = {df_filtrado_arg['provincia'].value_counts().sum()}")

df_filtrado_arg["provincia"].value_counts(dropna=False)


import matplotlib.pyplot as plt

conteo_provincia = (
    df_filtrado_arg["provincia"]
    .fillna("Sin provincia")
    .value_counts()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 10))
conteo_provincia.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Cantidad de registros por provincia")
ax.set_xlabel("Cantidad")
ax.set_ylabel("Provincia")
plt.tight_layout()
plt.show()

In [ ]:
# Analizamos la cantidad de palabras y el largo del texto en la columna "texto"

# Cantidad de palabras por registro
# Se reemplazan los valores nulos por cadenas vacías para evitar errores
palabras_texto = df_filtrado_arg["texto"].fillna("").str.split().str.len()

# Largo en caracteres por registro
largo_texto = df_filtrado_arg["texto"].fillna("").str.len()

# Agregamos estas columnas al DataFrame
df_filtrado_arg["n_palabras_texto"] = palabras_texto
df_filtrado_arg["longitud_texto"] = largo_texto

# Resumen estadístico
pd.DataFrame({
    "n_palabras_texto": palabras_texto,
    "longitud_texto": largo_texto
}).describe()

df_filtrado_arg.head(2)

In [ ]:
# ------------------------------------------------------------
# 1) Candidatas a variable objetivo
# ------------------------------------------------------------
candidatas = ["fuero", "tribunal", "materia", "jurisdiccion", "provincia", "pais"]

for col in candidatas:
    if col in df_sample_limpio.columns:
        s = df_sample_limpio[col]
        print(f"\n=== {col} ===")
        print("nulos:", s.isna().sum())
        print("cant_clases:", s.dropna().nunique())
        print(s.value_counts(dropna=False).head(15))

In [ ]:
# ------------------------------------------------------------
# 2) Resumen de calidad de cada candidata
# ------------------------------------------------------------
resumen = []

for col in candidatas:
    if col in df_sample_limpio.columns:
        s = df_sample_limpio[col].dropna()
        counts = s.value_counts()
        resumen.append({
            "columna": col,
            "nulos": df_sample_limpio[col].isna().sum(),
            "clases": counts.shape[0],
            "clase_mas_comun": counts.index[0] if not counts.empty else None,
            "freq_clase_mas_comun": int(counts.iloc[0]) if not counts.empty else None,
            "min_freq": int(counts.min()) if not counts.empty else None,
            "max_freq": int(counts.max()) if not counts.empty else None,
        })

pd.DataFrame(resumen).sort_values("clases", ascending=False)

In [ ]:
# ------------------------------------------------------------
# 3) Elegir una variable objetivo tentativa
# ------------------------------------------------------------
# Recomendación práctica: empezar por "fuero" si está completo y tiene suficiente cantidad
# de ejemplos por clase. Si no, probar "materia" o una versión agrupada.

target_cand = "materia" if "materia" in df_sample_limpio.columns else None

if target_cand is not None:
    s = df_sample_limpio[target_cand]
    print(f"\nTarget tentativa: {target_cand}")
    print(s.value_counts(dropna=False).head(20))

In [ ]:
import nltk
from nltk.corpus import stopwords

# 1. Descargar las stopwords (solo la primera vez)
nltk.download("stopwords")

In [ ]:
# 2. Cargar la lista de stopwords en español
stop_words = set(stopwords.words("spanish"))
stop_words.add("uuid")
stop_words.add("art") # agrego esta solo para analizar la salida. 

In [ ]:
# ------------------------------------------------------------
# 4) Ver si el texto aporta señal para esa variable
# ------------------------------------------------------------
# Para esto, podemos mirar palabras frecuentes por clase
# Primero, limpiamos y tokenizamos el texto
import re
from collections import Counter


def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = re.sub(r"<[^>]+>", " ", texto)
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)
    texto = re.sub(r"[^a-záéíóúúñ\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


# Lista de stopwords en español de NLTK
# Se usa stop_words que se definió en la celda anterior

def tokenizar_y_limpiar(texto):
    limpio = limpiar_texto(texto)
    tokens = [tok for tok in limpio.split() if tok not in stop_words and len(tok) > 2]
    return tokens


# Elegimos el target solo si existe en el DataFrame
def analizar_texto_por_clase(target_col):
    if target_col is None or target_col not in df_sample_limpio.columns:
        print("No se encontró una columna válida para usar como target.")
        return

    df_texto = df_sample_limpio[[target_col, "texto"]].copy()
    df_texto["texto_limpio"] = df_texto["texto"].apply(limpiar_texto)
    df_texto["tokens"] = df_texto["texto_limpio"].apply(tokenizar_y_limpiar)

    for clase in df_texto[target_col].dropna().astype(str).unique()[:10]:
        subset = df_texto[df_texto[target_col].astype(str) == clase]["tokens"]
        tokens_flat = [tok for token_list in subset for tok in token_list]
        counter = Counter(tokens_flat)
        print(f"\n--- {clase} ---")
        print(counter.most_common(20))


analizar_texto_por_clase(target_cand)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Gráfico para comparar categorías de materia
if "materia" in df_sample_limpio.columns:
    materias = (
        df_sample_limpio["materia"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .value_counts()
    )

    if not materias.empty:
        top_n = 15
        if len(materias) > top_n:
            top = materias.head(top_n)
            otras = pd.Series({"Otras": int(materias.iloc[top_n:].sum())})
            datos = pd.concat([top, otras])
        else:
            datos = materias

        datos = datos.sort_values(ascending=True)

        fig, ax = plt.subplots(figsize=(10, 6))
        datos.plot(kind="barh", color="steelblue", ax=ax)
        ax.set_title("Comparación de materias por cantidad de registros")
        ax.set_xlabel("Cantidad de registros")
        ax.set_ylabel("Materia")
        plt.tight_layout()
        plt.show()
    else:
        print("No hay valores de materia para graficar.")
else:
    print("La columna 'materia' no existe en el DataFrame.")


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

if "materia" in df_sample_limpio.columns and "texto" in df_sample_limpio.columns:
    df_texto = df_sample_limpio[["materia", "texto"]].copy()
    df_texto["materia"] = (
        df_texto["materia"]
        .astype(str)
        .str.strip()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )
    df_texto = df_texto.dropna(subset=["materia", "texto"])

    top_materias = (
        df_texto["materia"]
        .value_counts()
        .head(5)
        .index.tolist()
    )

    if top_materias:
        fig, axes = plt.subplots(len(top_materias), 1, figsize=(10, 3.2 * len(top_materias)))
        axes = axes.flatten()

        for ax, materia in zip(axes, top_materias):
            subset = df_texto[df_texto["materia"] == materia]
            tokens = [tok for texto in subset["texto"] for tok in tokenizar_y_limpiar(texto)]
            counter = Counter(tokens)
            top_words = counter.most_common(10)

            labels = [w for w, _ in top_words]
            values = [c for _, c in top_words]

            ax.barh(labels[::-1], values[::-1], color="seagreen")
            ax.set_title(f"Palabras más frecuentes - {materia}")
            ax.set_xlabel("Frecuencia")
            ax.invert_yaxis()

        plt.tight_layout()
        plt.show()
    else:
        print("No hay materias suficientes para graficar.")
else:
    print("No se encontraron las columnas 'materia' o 'texto'.")


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

if "materia" in df_sample_limpio.columns and "texto" in df_sample_limpio.columns:
    df_wc = df_sample_limpio[["materia", "texto"]].copy()
    df_wc["materia"] = (
        df_wc["materia"]
        .astype(str)
        .str.strip()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )
    df_wc = df_wc.dropna(subset=["materia", "texto"])

    top_materias = df_wc["materia"].value_counts().head(4).index.tolist()

    if top_materias:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        for ax, materia in zip(axes, top_materias):
            subset = df_wc[df_wc["materia"] == materia]
            texto = " ".join(subset["texto"].astype(str).tolist())
            tokens = tokenizar_y_limpiar(texto)
            texto_limpio = " ".join(tokens)

            wc = WordCloud(
                width=800,
                height=400,
                background_color="white",
                colormap="viridis"
            ).generate(texto_limpio)

            ax.imshow(wc, interpolation="bilinear")
            ax.set_title(f"Wordcloud - {materia}")
            ax.axis("off")

        for ax in axes[len(top_materias):]:
            ax.axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print("No hay materias suficientes para generar wordclouds.")
else:
    print("No se encontraron las columnas 'materia' o 'texto'.")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import matplotlib.pyplot as plt

if "materia" in df_sample_limpio.columns and "texto" in df_sample_limpio.columns:
    df_tfidf = df_sample_limpio[["materia", "texto"]].copy()
    df_tfidf["materia"] = (
        df_tfidf["materia"]
        .astype(str)
        .str.strip()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )
    df_tfidf = df_tfidf.dropna(subset=["materia", "texto"])

    top_materias = df_tfidf["materia"].value_counts().head(4).index.tolist()
    df_tfidf = df_tfidf[df_tfidf["materia"].isin(top_materias)]

    if not df_tfidf.empty:
        docs = []
        labels = []
        for materia in top_materias:
            subset = df_tfidf[df_tfidf["materia"] == materia]
            textos = subset["texto"].astype(str).tolist()
            docs.extend(textos)
            labels.extend([materia] * len(textos))

        vectorizer = TfidfVectorizer(
            stop_words=stop_words,
            ngram_range=(1, 2),
            max_features=100
        )
        X = vectorizer.fit_transform(docs)

        feature_names = vectorizer.get_feature_names_out()
        feature_scores = X.toarray()

        # Promedio por clase para resaltar términos distintivos
        scores_by_label = {}
        for materia in top_materias:
            idx = [i for i, label in enumerate(labels) if label == materia]
            if idx:
                scores_by_label[materia] = feature_scores[idx].mean(axis=0)
            else:
                scores_by_label[materia] = np.zeros(X.shape[1])

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        for ax, materia in zip(axes, top_materias):
            scores = scores_by_label[materia]
            top_idx = np.argsort(scores)[-8:][::-1]
            top_terms = [(feature_names[i], scores[i]) for i in top_idx]

            terms = [t for t, _ in top_terms]
            vals = [v for _, v in top_terms]

            ax.barh(terms[::-1], vals[::-1], color="tomato")
            ax.set_title(f"TF-IDF top terms - {materia}")
            ax.set_xlabel("Score TF-IDF")
            ax.invert_yaxis()

        for ax in axes[len(top_materias):]:
            ax.axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print("No hay datos suficientes para TF-IDF.")
else:
    print("No se encontraron las columnas 'materia' o 'texto'.")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import matplotlib.pyplot as plt

if "materia" in df_sample_limpio.columns and "texto" in df_sample_limpio.columns:
    df_tfidf = df_sample_limpio[["materia", "texto"]].copy()
    df_tfidf["materia"] = (
        df_tfidf["materia"]
        .astype(str)
        .str.strip()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )
    df_tfidf = df_tfidf.dropna(subset=["materia", "texto"])

    top_materias = df_tfidf["materia"].value_counts().head(10).index.tolist()
    df_tfidf = df_tfidf[df_tfidf["materia"].isin(top_materias)]

    if not df_tfidf.empty:
        docs = []
        labels = []
        for materia in top_materias:
            subset = df_tfidf[df_tfidf["materia"] == materia]
            textos = subset["texto"].astype(str).tolist()
            docs.extend(textos)
            labels.extend([materia] * len(textos))

        stop_words_list = list(stop_words)
        vectorizer = TfidfVectorizer(
            stop_words=stop_words_list,
            ngram_range=(1, 2),
            max_features=100
        )
        X = vectorizer.fit_transform(docs)

        feature_names = vectorizer.get_feature_names_out()
        feature_scores = X.toarray()

        scores_by_label = {}
        for materia in top_materias:
            idx = [i for i, label in enumerate(labels) if label == materia]
            if idx:
                scores_by_label[materia] = feature_scores[idx].mean(axis=0)
            else:
                scores_by_label[materia] = np.zeros(X.shape[1])

        fig, axes = plt.subplots(4, 4, figsize=(12, 10))
        axes = axes.flatten()

        for ax, materia in zip(axes, top_materias):
            scores = scores_by_label[materia]
            top_idx = np.argsort(scores)[-8:][::-1]
            top_terms = [(feature_names[i], scores[i]) for i in top_idx]

            terms = [t for t, _ in top_terms]
            vals = [v for _, v in top_terms]

            ax.barh(terms[::-1], vals[::-1], color="tomato")
            ax.set_title(f"TF-IDF top terms - {materia}")
            ax.set_xlabel("Score TF-IDF")
            ax.invert_yaxis()

        for ax in axes[len(top_materias):]:
            ax.axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print("No hay datos suficientes para TF-IDF.")
else:
    print("No se encontraron las columnas 'materia' o 'texto'.")


In [101]:
for col in df_sample_limpio.columns:
    print(col)

numero-sumario
materia
sumario
descriptores
referencias-normativas
texto
fuente
analista
responsable
fecha
tipo-tribunal
instancia
jurisdiccion
provincia
caratula
fecha-alta
fecha-mod
uid-alta
uid-mod
timestamp
timestamp-m
timestamp-alta
id-infojus
fecha-umod
titulo
guid
numero-fallo
tribunal
pais
tipo-fallo
localidad
magistrados
actor
demandado
sobre
sumarios-relacionados
texto-doc
sala
numero-interno
tribunal-origen
publicacion
texto-completo
numero-camara
citas
jurisprudencia-vinculada
sintesis
hechos
identificacion-plenario
fecha-alta_dt
fecha-mod_dt
fecha-alta_mes_num
fecha-alta_mes_nom
fecha-mod_mes_num
fecha-mod_mes_nom


In [ ]:
# Análisis de fechas: fecha-alta y fecha-mod
import pandas as pd
import matplotlib.pyplot as plt

for col in ["fecha-alta", "fecha-mod"]:
    if col in df_sample_limpio.columns:
        s = df_sample_limpio[col]
        print(f"\n== {col} ==")
        print("nulos:", s.isna().sum())
        print("no nulos:", s.notna().sum())
        print("valores únicos (primeros 10):")
        print(s.dropna().astype(str).head(10).tolist())
        print("\nFormato de ejemplo:")
        print(s.dropna().astype(str).head(5).to_string(index=False))

        # Convertimos a tipo fecha
        df_sample_limpio[f"{col}_dt"] = pd.to_datetime(s, errors="coerce")

        # Resumen por año
        serie_anio = df_sample_limpio[f"{col}_dt"].dt.year
        print("\nConteo por año:")
        print(serie_anio.value_counts().sort_index())

        # Gráfico por año
        counts = serie_anio.value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(10, 4))
        counts.plot(kind="bar", ax=ax, color="steelblue")
        ax.set_title(f"Distribución por año - {col}")
        ax.set_xlabel("Año")
        ax.set_ylabel("Cantidad")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print(f"La columna {col} no existe")


In [ ]:
# Análisis de estacionalidad por mes: fecha-alta y fecha-mod
import pandas as pd
import matplotlib.pyplot as plt

for col in ["fecha-alta", "fecha-mod"]:
    if col in df_sample_limpio.columns:
        col_dt = f"{col}_dt"
        if col_dt not in df_sample_limpio.columns:
            df_sample_limpio[col_dt] = pd.to_datetime(df_sample_limpio[col], errors="coerce")

        s = df_sample_limpio[col_dt]

        # Extraer mes y nombre del mes
        df_sample_limpio[f"{col}_mes_num"] = s.dt.month
        df_sample_limpio[f"{col}_mes_nom"] = s.dt.month_name(locale="es_ES")

        conteo_mes = (
            df_sample_limpio[f"{col}_mes_num"]
            .value_counts()
            .sort_index()
        )

        print(f"\n== {col} ==")
        print(conteo_mes)

        # Ordenar meses del 1 al 12
        orden_meses = [
            "enero", "febrero", "marzo", "abril", "mayo", "junio",
            "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
        ]
        etiquetas = [m for m in orden_meses if m in df_sample_limpio[f"{col}_mes_nom"].dropna().astype(str).str.lower().tolist()]

        # Conteo por nombre de mes
        counts = (
            df_sample_limpio[f"{col}_mes_nom"]
            .dropna()
            .str.lower()
            .value_counts()
            .reindex(orden_meses, fill_value=0)
        )

        fig, ax = plt.subplots(figsize=(10, 4))
        counts.plot(kind="bar", ax=ax, color="sandybrown")
        ax.set_title(f"Estacionalidad por mes - {col}")
        ax.set_xlabel("Mes")
        ax.set_ylabel("Cantidad")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print(f"La columna {col} no existe")


In [102]:
# Análisis de columnas de identificador: uid-alta, uid-mod y guid
import pandas as pd
import matplotlib.pyplot as plt

cols = [c for c in ["uid-alta", "uid-mod", "guid", "fecha-alta", "fecha-mod"] if c in df_sample_limpio.columns]
df_ids = df_sample_limpio[cols].copy()

# Normalizamos los IDs y convertimos fechas
for col in [c for c in ["uid-alta", "uid-mod", "guid"] if c in df_ids.columns]:
    df_ids[col] = df_ids[col].astype("string")

for col in [c for c in ["fecha-alta", "fecha-mod"] if c in df_ids.columns]:
    df_ids[f"{col}_dt"] = pd.to_datetime(df_sample_limpio[col], errors="coerce")

print("Columnas disponibles para el análisis:")
print(df_ids.columns.tolist())

# Resumen básico de cada identificador
for col in [c for c in ["uid-alta", "uid-mod", "guid"] if c in df_ids.columns]:
    s = df_ids[col]
    print(f"\n== {col} ==")
    print("nulos:", s.isna().sum())
    print("no nulos:", s.notna().sum())
    print("únicos:", s.dropna().nunique())
    print(s.dropna().value_counts().head(10))

# Comparación directa entre los identificadores
if all(c in df_ids.columns for c in ["uid-alta", "uid-mod", "guid"]):
    df_ids["uid_alta_igual_uid_mod"] = (
        df_ids["uid-alta"].fillna("").astype(str) == df_ids["uid-mod"].fillna("").astype(str)
    )
    df_ids["uid_alta_igual_guid"] = (
        df_ids["uid-alta"].fillna("").astype(str) == df_ids["guid"].fillna("").astype(str)
    )
    df_ids["uid_mod_igual_guid"] = (
        df_ids["uid-mod"].fillna("").astype(str) == df_ids["guid"].fillna("").astype(str)
    )

    print("\nComparaciones binarias:")
    print("uid-alta == uid-mod:", df_ids["uid_alta_igual_uid_mod"].mean())
    print("uid-alta == guid:", df_ids["uid_alta_igual_guid"].mean())
    print("uid-mod == guid:", df_ids["uid_mod_igual_guid"].mean())

# ¿Los IDs parecen repetirse en el tiempo?
for col in [c for c in ["uid-alta", "uid-mod", "guid"] if c in df_ids.columns]:
    agg = (
        df_ids[[col, "fecha-alta_dt", "fecha-mod_dt"]]
        .dropna(subset=[col])
        .groupby(col, dropna=False)
        .agg(
            n_registros=("fecha-alta_dt", "size"),
            primer_fecha_alta=("fecha-alta_dt", "min"),
            ultima_fecha_alta=("fecha-alta_dt", "max"),
            primer_fecha_mod=("fecha-mod_dt", "min"),
            ultima_fecha_mod=("fecha-mod_dt", "max"),
        )
        .sort_values("n_registros", ascending=False)
    )
    print(f"\nResumen por {col} (primeros 10):")
    print(agg.head(10))

# Por si querés ver si el ID cambia según la fecha de alta/modificación
for col in [c for c in ["uid-alta", "uid-mod", "guid"] if c in df_ids.columns]:
    resumen_fecha = (
        df_ids[[col, "fecha-alta_dt"]]
        .dropna(subset=[col, "fecha-alta_dt"])
        .groupby(col)
        .agg(
            n_registros=("fecha-alta_dt", "size"),
            primera_fecha=("fecha-alta_dt", "min"),
            ultima_fecha=("fecha-alta_dt", "max"),
        )
        .sort_values(["ultima_fecha", "n_registros"], ascending=False)
        .head(10)
    )
    print(f"\nTop 10 por fecha de alta para {col}:")
    print(resumen_fecha)


Columnas disponibles para el análisis:
['uid-alta', 'uid-mod', 'guid', 'fecha-alta', 'fecha-mod', 'fecha-alta_dt', 'fecha-mod_dt']

== uid-alta ==
nulos: 1039
no nulos: 7709
únicos: 3
uid-alta
SAUID      4646
ABM        3041
VB-Auto      22
Name: count, dtype: int64[pyarrow]

== uid-mod ==
nulos: 1039
no nulos: 7709
únicos: 4
uid-mod
ABM        4375
SASAIJ     2437
SAUID       875
VB-Auto      22
Name: count, dtype: int64[pyarrow]

== guid ==
nulos: 0
no nulos: 8748
únicos: 8748
guid
123456789-0abc-defg9415-000ssoiramus    1
123456789-0abc-defg5685-000csoiramus    1
123456789-0abc-defg3735-300bsoiramus    1
123456789-701-0831-2ots-eupmocsollaf    1
123456789-0abc-defg4694-400bsoiramus    1
123456789-352-0226-0ots-eupmocsollaf    1
123456789-0abc-defg1420-040csoiramus    1
123456789-0abc-defg1959-700asoiramus    1
123456789-0abc-defg8938-000csoiramus    1
123456789-0abc-defg1020-000rsoiramus    1
Name: count, dtype: int64[pyarrow]

Comparaciones binarias:
uid-alta == uid-mod: 0.50331504

In [ ]:
# Análisis de fechas por materia: foco en fecha-mod y estacionalidad
import pandas as pd
import matplotlib.pyplot as plt

if "materia" in df_sample_limpio.columns and "fecha-mod" in df_sample_limpio.columns:
    df_mat = df_sample_limpio[["materia", "fecha-mod", "fecha-alta"]].copy()

    # Limpiar y convertir fechas
    df_mat["materia"] = (
        df_mat["materia"]
        .astype(str)
        .str.strip()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )
    df_mat = df_mat.dropna(subset=["materia"])

    for col in ["fecha-mod", "fecha-alta"]:
        df_mat[f"{col}_dt"] = pd.to_datetime(df_mat[col], errors="coerce")

    # Top materias por cantidad de registros
    top_materias = df_mat["materia"].value_counts().head(6).index.tolist()
    df_mat_top = df_mat[df_mat["materia"].isin(top_materias)]

    print("Materias analizadas:")
    print(top_materias)

    # 1) Conteo por mes para fecha-mod, desagregado por materia
    fig, ax = plt.subplots(figsize=(12, 5))
    for materia in top_materias:
        subset = df_mat_top[df_mat_top["materia"] == materia]
        counts = (
            subset["fecha-mod_dt"]
            .dropna()
            .dt.month_name(locale="es_ES")
            .str.lower()
            .value_counts()
            .reindex([
                "enero", "febrero", "marzo", "abril", "mayo", "junio",
                "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
            ], fill_value=0)
        )
        counts.plot(kind="bar", ax=ax, alpha=0.7, label=materia)

    ax.set_title("Distribución mensual de fecha-mod por materia")
    ax.set_xlabel("Mes")
    ax.set_ylabel("Cantidad de registros")
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # 2) Comparación de la proporción mensual por materia
    monthly = []
    for materia in top_materias:
        subset = df_mat_top[df_mat_top["materia"] == materia]
        counts = (
            subset["fecha-mod_dt"]
            .dropna()
            .dt.month_name(locale="es_ES")
            .str.lower()
            .value_counts()
            .reindex([
                "enero", "febrero", "marzo", "abril", "mayo", "junio",
                "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
            ], fill_value=0)
        )
        counts = counts / counts.sum() if counts.sum() > 0 else counts
        monthly.append(pd.Series(counts, name=materia))

    monthly_df = pd.concat(monthly, axis=1).T
    print("\nProporción mensual por materia (fecha-mod):")
    print(monthly_df)

    # 3) Resumen por año y materia para fecha-mod
    resumen_anio_materia = (
        df_mat_top.dropna(subset=["fecha-mod_dt"])
        .groupby(["materia", df_mat_top["fecha-mod_dt"].dt.year])
        .size()
        .unstack(fill_value=0)
    )
    print("\nConteo por año y materia (fecha-mod):")
    print(resumen_anio_materia)

else:
    print("No se encontraron las columnas 'materia' y 'fecha-mod'.")
